<a href="https://colab.research.google.com/github/Jayku88/22AIE301_Probabilistic_Reasoning/blob/main/22AIE301_Lab_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 22AIE301 - Probabilistic Reasoning
## Lab 05 - Bayesian Networks I: Motivation and Factorization

| | |
|---|---|
| **Name** | Vishnu.S |
| **Roll No** | AM.SC.U4AIE24058 |
| **Date** |22-07-2026|
| **Lab Slot** | _____________ |

---

**Based on:** Lecture 5 - Bayesian Networks I: Motivation and Factorization

**Instructions**
- This lab has **6 hands-on exercises**, to be completed in **1 hour 30 minutes**.
- We use the `pgmpy` library to build and query the Bayesian networks introduced in lecture (student performance, naive Bayes spam filter, sprinkler network).
- Each exercise has starter code. Fill every `___` blank to make the code run correctly.
- Every task has an `assert` check right after it. If your code is correct, the cell runs silently; if not, you'll see an `AssertionError`.
- Exercises 1 and 6 use parameters/structures derived from **your own roll number** — your numbers will differ from your classmates'.
- Run cells top to bottom. Do not skip the Setup cell.
- Show your outputs to the facilitator before leaving the lab.


## Setup: Run this cell first before anything else


In [17]:
!pip install pgmpy -q

import itertools
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

ROLL = 58    # <-- Replace with your roll number (last 3 digits)
np.random.seed(ROLL)

# Personalized parameters for Exercise 1, derived from your roll number seed.
# Do NOT hardcode these values anywhere else in the notebook --
# always refer to n_vars, d_card, k_max by name.
n_vars = 15 + (ROLL % 10)     # number of variables in the naive joint
d_card = 2 + (ROLL % 3)       # cardinality per variable,
k_max  = 1 + (ROLL % 4)       # max parents per variable,

print(f"n_vars={n_vars}, d_card={d_card}, k_max={k_max}")


n_vars=23, d_card=3, k_max=3


---
## Exercise 1 · Naive vs. Compact Parameter Counts

Recall from lecture: for $n$ variables of cardinality $d$, the **naive** joint table needs
$$d^n - 1$$
free parameters. If we instead assume each variable has **at most $k$ parents**, the
**compact** (Bayesian network) representation needs only
$$n\, d^{k}(d-1)$$
free parameters (uniform-cardinality, uniform-parent-count case).

Use `n_vars`, `d_card`, `k_max` from the Setup cell - these were derived from your roll number.

**Tasks**

| # | Task |
|---|------|
| 1a | Compute the naive parameter count. |
| 1b | Compute the compact (Bayesian network) parameter count. |
| 1c | Compute and print the reduction factor. |


In [18]:
# -- 1a: Naive parameter count -------------------------------
naive_params = d_card ** n_vars - 1                       # d_card, n_vars

# -- 1b: Compact (Bayesian network) parameter count ----------
compact_params = n_vars * (d_card ** k_max) * (d_card - 1)      # n_vars, d_card, k_max, d_card

# -- 1c: Reduction factor -------------------------------------
reduction = naive_params / compact_params

print(f"Naive parameters   : {naive_params:,}")
print(f"Compact parameters : {compact_params:,}")
print(f"Reduction factor    : {reduction:,.1f}x")

assert compact_params < naive_params, "Compact representation should be far smaller than naive for these settings"


Naive parameters   : 94,143,178,826
Compact parameters : 1,242
Reduction factor    : 75,799,660.9x


---
## Exercise 2 · Building the Student Network in `pgmpy`

Recall the student performance network from lecture:
$$p(l,g,i,d,s) = p(l\mid g)\,p(g\mid i,d)\,p(s\mid i)\,p(i)\,p(d)$$
with edges $D\to G$, $I\to G$, $I\to S$, $G\to L$.

**CPTs** (exactly as shown in lecture):

| | $d^0$ | $d^1$ |
|---|---|---|
| $p(d)$ | 0.6 | 0.4 |

| | $i^0$ | $i^1$ |
|---|---|---|
| $p(i)$ | 0.7 | 0.3 |

| | $g^1$ | $g^2$ | $g^3$ |
|---|---|---|---|
| $i^0,d^0$ | 0.30 | 0.40 | 0.30 |
| $i^0,d^1$ | 0.05 | 0.25 | 0.70 |
| $i^1,d^0$ | 0.90 | 0.08 | 0.02 |
| $i^1,d^1$ | 0.50 | 0.30 | 0.20 |

| | $s^0$ | $s^1$ |
|---|---|---|
| $i^0$ | 0.95 | 0.05 |
| $i^1$ | 0.20 | 0.80 |

| | $l^0$ | $l^1$ |
|---|---|---|
| $g^1$ | 0.10 | 0.90 |
| $g^2$ | 0.40 | 0.60 |
| $g^3$ | 0.99 | 0.01 |

**Task.** `pgmpy`'s `TabularCPD` stores each column of a table as a *state* of the variable
(state 0, state 1, ...), and columns of the values matrix correspond to parent-configurations
in the order the `evidence` list is given. Fill in the missing pieces below so that the model
passes `check_model()`.


In [19]:
student = DiscreteBayesianNetwork([("D", "G"), ("I", "G"), ("I", "S"), ("G", "L")])

cpd_d = TabularCPD("D", 2, [[0.6], [0.4]])
cpd_i = TabularCPD("I", 2, [[0.7], [0.3]])

cpd_g = TabularCPD("G", 3,                                       # cardinality of G (g1,g2,g3)
    [[0.30, 0.05, 0.90, 0.50],
     [0.40, 0.25, 0.08, 0.30],
     [0.30, 0.70, 0.02, 0.20]],
    evidence=["I", "D"], evidence_card=[2,2])                 # parents of G, in order I, D

cpd_s = TabularCPD("S", 2,
    [[0.95, 0.20],
     [0.05, 0.80]],
    evidence=["I"], evidence_card=[2])                           # parent of S

cpd_l = TabularCPD("L", 2,
    [[0.10, 0.40, 0.99],
     [0.90, 0.60, 0.01]],
    evidence=["G"], evidence_card=[3])                           # parent of L

student.add_cpds(cpd_d, cpd_i, cpd_g, cpd_s, cpd_l)
assert student.check_model(), "Model is not valid -- check your CPDs and evidence"
print("Student network is valid.")


Student network is valid.


---
## Exercise 3 · A Fully Worked Joint Query

Recall from lecture: $p(d^0,i^1,g^1,s^1,l^0) = p(d^0)\cdot p(i^1)\cdot p(g^1\mid i^1,d^0)\cdot p(s^1\mid i^1)\cdot p(l^0\mid g^1) = 0.01296$.

**Tasks**

| # | Task |
|---|------|
| 3a | Reproduce this computation by hand, reading the five factors off the CPDs / lecture tables. |
| 3b | Cross-check it by taking the **factor product** of all five CPDs in `pgmpy` and reading off the same joint assignment. |
| 3c | Compute the marginal $p(l^0)$ using `VariableElimination`. |


In [20]:
infer_student = VariableElimination(student)

# -- 3a: Hand-computed joint p(d0, i1, g1, s1, l0) -------------
p_d0        = cpd_d.get_values()[0, 0]
p_i1        = cpd_i.get_values()[1, 0]        # row index for i1
p_g1_i1_d0  = 0.90
p_s1_i1     = 0.80
p_l0_g1     = 0.10

joint_hand = p_d0 * p_i1 * p_g1_i1_d0 * p_s1_i1 * p_l0_g1         # multiply all five factors, in order

print(f"Hand-computed joint = {joint_hand:.5f}")

# -- 3b: Cross-check via pgmpy's factor product -----------------
joint_factor = cpd_d.to_factor()
for cpd in [cpd_i, cpd_g, cpd_s, cpd_l]:
    joint_factor = joint_factor.product(cpd.to_factor(), inplace=False)        # cpd.to_factor()

joint_pgmpy = joint_factor.get_value(D=0, I=1, G=0, S=1, L=0)
print(f"pgmpy joint          = {joint_pgmpy:.5f}")

assert abs(joint_hand - joint_pgmpy) < 1e-6, "Hand and pgmpy computations disagree"

# -- 3c: Marginal probability of a weak letter -------------------
marg_l = infer_student.query(variables=["L"], show_progress=False)  # "L"
print(marg_l)


Hand-computed joint = 0.01296
pgmpy joint          = 0.01296
+------+----------+
| L    |   phi(L) |
+======+==========+
| L(0) |   0.4977 |
+------+----------+
| L(1) |   0.5023 |
+------+----------+


---
## Exercise 4 · Naive Bayes as a Bayesian Network (Spam Filter)

Recall the star-shaped naive Bayes network: $C\to X_j$ for every word $j$, with
$p(c,x_1,\ldots,x_m) = p(c)\prod_j p(x_j\mid c)$. From lecture, $p(\text{spam})=0.4$ and:

| Word present | $p(\cdot\mid\text{spam})$ | $p(\cdot\mid\text{ham})$ |
|---|---|---|
| "free" | 0.8 | 0.1 |
| "winner" | 0.6 | 0.05 |
| "meeting" | 0.1 | 0.5 |

Here, state `0` of `C` is **spam**, state `1` is **ham**; state `0` of each word variable is
**present**, state `1` is **absent**.

**Tasks**

| # | Task |
|---|------|
| 4a | Build the naive Bayes network structure and CPDs. |
| 4b | Personalize the evidence: derive which words are present/absent from your `ROLL`. |
| 4c | Query $p(C\mid \text{evidence})$ and read off the classification. |


In [21]:
nb = DiscreteBayesianNetwork([("C", "free"), ("C", "winner"), ("C", "meeting")])   # C->winner, C->meeting

cpd_c2 = TabularCPD("C", 2, [[0.4], [0.6]])                              # 0=spam, 1=ham
cpd_free = TabularCPD("free", 2,
    [[0.8, 0.1], [0.2, 0.9]], evidence=["C"], evidence_card=[2])       # "C", 2
cpd_winner = TabularCPD("winner", 2,
    [[0.6, 0.05], [0.4, 0.95]], evidence=["C"], evidence_card=[2])
cpd_meeting = TabularCPD("meeting", 2,
    [[0.1, 0.5], [0.9, 0.5]], evidence=["C"], evidence_card=[2])

nb.add_cpds(cpd_c2, cpd_free, cpd_winner, cpd_meeting)
assert nb.check_model(), "Model is not valid -- check your CPDs and evidence"

# -- 4b: Personalize the evidence pattern from your roll number --
free_state    = ROLL % 2            # 0 = present, 1 = absent
winner_state  = (ROLL // 2) % 2
meeting_state = (ROLL // 3) % 2   # what divisor keeps this independent of free_state and winner_state?

print("Evidence (0=present, 1=absent):",
      f"free={free_state}, winner={winner_state}, meeting={meeting_state}")

# -- 4c: Query the posterior over C given this evidence -----------
infer_nb = VariableElimination(nb)
posterior = infer_nb.query(
    variables=["C"],                                                    # "C"
    evidence={"free": free_state, "winner": winner_state, "meeting": meeting_state},               # the three states above
    show_progress=False)
print(posterior)


Evidence (0=present, 1=absent): free=0, winner=1, meeting=1
+------+----------+
| C    |   phi(C) |
+======+==========+
| C(0) |   0.8017 |
+------+----------+
| C(1) |   0.1983 |
+------+----------+


---
## Exercise 5 · The Sprinkler Network

Recall the sprinkler network: $p(C,S,R,W) = p(C)\,p(S\mid C)\,p(R\mid C)\,p(W\mid S,R)$, with
lecture's verified answers $p(C{=}T,S{=}T,R{=}T,W{=}T)=0.0396$ and $p(S{=}T)=0.30$.

**Tasks**

| # | Task |
|---|------|
| 5a | Build the sprinkler network with the CPTs from lecture. |
| 5b | Compute the joint $p(C{=}T,S{=}T,R{=}T,W{=}T)$ via factor product and check it against 0.0396. |
| 5c | Compute the marginal $p(S{=}T)$ via `VariableElimination` and check it against 0.30. |


In [22]:
sprinkler = DiscreteBayesianNetwork([("C", "S"), ("C", "R"), ("S", "W"), ("R", "W")])

cpd_c3 = TabularCPD("C", 2, [[0.5], [0.5]])
cpd_s3 = TabularCPD("S", 2,
    [[0.1, 0.5], [0.9, 0.5]], evidence=["C"], evidence_card=[2])       # "C", 2
cpd_r3 = TabularCPD("R", 2,
    [[0.8, 0.2], [0.2, 0.8]], evidence=["C"], evidence_card=[2])       # "C", 2
cpd_w3 = TabularCPD("W", 2,
    [[0.99, 0.90, 0.90, 0.00],
     [0.01, 0.10, 0.10, 1.00]],
    evidence=["S", "R"], evidence_card=[2, 2])                      # parents of W: S, R

sprinkler.add_cpds(cpd_c3, cpd_s3, cpd_r3, cpd_w3)
assert sprinkler.check_model(), "Model is not valid -- check your CPDs and evidence"

# -- 5b: Joint via factor product ----------------------------------
joint_factor2 = cpd_c3.to_factor()
for cpd in [cpd_s3, cpd_r3, cpd_w3]:
    joint_factor2 = joint_factor2.product(cpd.to_factor(), inplace=False)            # cpd.to_factor()

joint_sprinkler = joint_factor2.get_value(C=0, S=0, R=0, W=0)
print(f"p(C=T,S=T,R=T,W=T) = {joint_sprinkler:.4f}")
assert abs(joint_sprinkler - 0.0396) < 1e-4                                 # 0.0396

# -- 5c: Marginal via VariableElimination --------------------------
infer_sprinkler = VariableElimination(sprinkler)
marg_s = infer_sprinkler.query(variables=["S"], show_progress=False)     # "S"
print(marg_s)


p(C=T,S=T,R=T,W=T) = 0.0396
+------+----------+
| S    |   phi(S) |
+======+==========+
| S(0) |   0.3000 |
+------+----------+
| S(1) |   0.7000 |
+------+----------+


---
## Exercise 6 · Personalized Mixed-Cardinality Parameter Count

Recall the Student Network mixed-cardinality parameter count from lecture:
$$\text{Total} = \sum_{i=1}^n (d_i-1)\!\!\prod_{x_j\in\mathrm{pa}(x_i)}\!\! d_j$$

The cell below deterministically generates, from your `ROLL`, a random 6-node DAG (edges only
from a lower-indexed node to a higher-indexed one) and a random cardinality ($2$ or $3$) for
each node. No blanks in the generator — just run it.


In [23]:
def generate_personal_bn_structure(seed, n_nodes=6, edge_prob=0.35):
    '''Generates a random DAG (edges only from lower- to higher-indexed node) and a
    random cardinality (2 or 3) for each node, deterministic given seed.'''
    rng = np.random.RandomState(seed)
    nodes = [f"X{i}" for i in range(n_nodes)]
    edges = []
    for i, j in itertools.combinations(range(n_nodes), 2):
        if rng.random() < edge_prob:
            edges.append((nodes[i], nodes[j]))
    cardinalities = {node: int(rng.choice([2, 3])) for node in nodes}
    return nodes, edges, cardinalities

my_nodes, my_edges, my_card = generate_personal_bn_structure(ROLL)
print("Nodes:", my_nodes)
print("Edges:", my_edges)
print("Cardinalities:", my_card)


Nodes: ['X0', 'X1', 'X2', 'X3', 'X4', 'X5']
Edges: [('X0', 'X4'), ('X1', 'X2'), ('X1', 'X3'), ('X3', 'X4')]
Cardinalities: {'X0': 3, 'X1': 2, 'X2': 2, 'X3': 3, 'X4': 3, 'X5': 2}


**Tasks**

| # | Task |
|---|------|
| 6a | Build the parent-set dictionary from the edge list. |
| 6b | Compute the total BN free-parameter count using the formula above. |
| 6c | Compute the naive joint-table parameter count for the same variables and compare. |


In [11]:
# -- 6a: Parent-set dictionary --------------------------------------
parents = {node: [] for node in my_nodes}
for u, v in my_edges:
    parents[u].append(v)                       # v, u  (u is a parent of v)

# -- 6b: Total BN free parameters -----------------------------------
total_params = 0
for node in my_nodes:
    config_count = 1
    for p in parents[node]:
        config_count *= my_card[p]                      # my_card
    total_params += (my_card[node] - 1) * config_count   # my_card

print(f"Total BN free parameters: {total_params}")

# -- 6c: Naive joint-table parameters, same variables ----------------
naive_total = 1
for node in my_nodes:
    naive_total *= my_card[node]                         # my_card
naive_total -= 1

print(f"Naive joint parameters  : {naive_total}")
print(f"Reduction factor         : {naive_total / total_params:,.1f}x")

assert total_params <= naive_total, "Your personalized BN should not need more parameters than the naive table"


Total BN free parameters: 22
Naive joint parameters  : 215
Reduction factor         : 9.8x


---
## Section 7 · Reflection Questions (Theory - answer in the markdown cell below, no code needed)

1. In Exercise 1, what happens to the reduction factor as `k_max` increases toward `n_vars`? Explain why, referencing the "fully connected DAG" row of the lecture's parent-count table.
2. In Exercise 2, why does `p(g|i,d)` need **4** parent configurations while `p(s|i)` needs only **2**? What general rule connects the number of parent configurations to the parent set?
3. In Exercise 4, is the naive Bayes conditional-independence assumption (words independent given spam/ham) literally true? Why does the model still work well for classification despite this?


### Your Answers — Section 7

**Q1:**

*(write your answer here)*
As kmax tends to n_wars the reduction factor will go to zero.

**Q2:**

*(write your answer here)*
p(g|i,d) has two binary output parents in which it will give 2^2 diffrent outputs while p(s|i) has one binary output parent which it will have 2^1 diffrent outputs. The genera rule is that the number of configurations is equal to the the product of number of possible states of each parent node.

**Q3:**

*(write your answer here)*
No the words are independent assumption  are not literally true.
Naive Bayes is designed to choose a label. Even though the conditional independence assumption skews the calculated probabilities (often pushing them artificially close to 0% or 100%), the relative ranking of the classes usually remains unchanged.


---
## Section 8 · Conceptual & Real-World Application Questions (Theory - you may use the internet)

These questions are **not** answered by running code - they test your conceptual
understanding of *why* and *when* Bayesian networks are used. You are encouraged
to use the internet (textbooks, Wikipedia, papers) to research Section 8, but
write answers in **your own words** and **cite your source**.

1. Give one real-world domain (other than spam filtering, student performance, or the
   sprinkler example) where a Bayesian network's compactness assumption plausibly holds,
   and name the variables and edges you would use.
2. Explain, in your own words, why a Bayesian network requires the graph to be a **DAG**
   (no directed cycles), using the $X\to Y\to X$ counterexample from lecture as a starting point.
3. What is the difference between *associational*, *interventional*, and *counterfactual*
   reasoning? Which of these does a Bayesian network's joint factorization alone support
   without further causal assumptions?
4. What is **modularity** of CPDs, and why does it make learning and editing a Bayesian
   network easier than editing a naive joint table? Give one concrete example.
5. Look up one real published application of Bayesian networks (medicine, genetics, robotics,
   finance, etc.). Briefly describe the variables used and cite your source.
6. In Exercise 6, explain in your own words why a densely connected DAG can sometimes need
   *more* parameters than the naive joint table, referencing the lecture's 19-parent row.


### Your Answers — Section 8

**Q1:**

*(write your answer here)*

**Q2:**

*(write your answer here)*

**Q3:**

*(write your answer here)*

**Q4:**

*(write your answer here)*

**Q5:**

*(write your answer here — include your source)*

**Q6:**

*(write your answer here)*


---
## Submission Checklist
-  All 6 hands-on exercises are complete and all cells produce output without errors.
-  Your **Roll No** is set correctly in the Setup cell (`ROLL`).
-  `student.check_model()`, `nb.check_model()`, and `sprinkler.check_model()` all pass.
-  Exercise 3 and Exercise 5's `assert` checks pass (hand vs. `pgmpy` agree; sprinkler joint/marginal match lecture values).
-  Section 7 (Reflection Questions) and Section 8 (Conceptual & Application Questions) are answered in your own words.
-  Save the notebook: **File → Save** (or Ctrl+S).

**Rename the file as:** `Lab05_<YourRollNo>.ipynb` and convert to PDF before submitting.
